# T34 - 债券新发行

## 第2章：环境配置与初始化

---

### 环境要求

- Python 3.8+
- MySQL 数据库
- Wind API (可选)
- 同花顺iFinD API (可选)

### 导入依赖库

In [ ]:
# 标准库
import os
import sys
import datetime
from datetime import datetime, timedelta
import logging
import warnings

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text, create_engine, inspect
import sqlalchemy.pool as pool

# 忽略警告
warnings.filterwarnings('ignore')

print("依赖库导入成功")

### 配置参数

使用环境变量获取数据库连接配置和API凭证，避免硬编码密码。

In [ ]:
# 从环境变量获取数据库配置
# 如果环境变量未设置，使用默认值（仅用于开发环境）

# MySQL配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')  # 从环境变量获取
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'yq')

# Wind API配置
WIND_USERNAME = os.environ.get('WIND_USERNAME', '')
WIND_PASSWORD = os.environ.get('WIND_PASSWORD', '')

# 同花顺iFinD API配置
IFIND_USERNAME = os.environ.get('IFIND_USERNAME', '')
IFIND_PASSWORD = os.environ.get('IFIND_PASSWORD', '')

print("配置参数加载完成")

### 业务参数配置

In [ ]:
# 债券到期数据采集配置
BOND_MATURITY_CONFIG = {
    'days_ahead': 7,  # 获取未来7天的数据
    'table_name': '债券到期',
    'database': 'yq'
}

# 债券新发行数据采集配置
BOND_ISSUE_CONFIG = {
    'days_start': 1,   # 从明天开始
    'days_end': 30,    # 到未来30天
    'table_name': '债券新发行1',
    'database': 'yq'
}

# 债券类型配置 (Wind API)
BOND_TYPES = [
    # (Wind类型代码, 中文名称, 是否城投债)
    ('governmentbonds', '国债', '否'),
    ('centralbankbills', '央行票据', '否'),
    ('cds', '存单', '否'),
    ('commercialbankbonds', '普通金融债', '否'),
    ('policybankbonds', '政策银行债', '否'),
    ('commercialbanksubordinatedbonds', '商业银行次级债券', '否'),
    ('insurancecompanybonds', '保险债', '否'),
    ('corporatebondsofsecuritiescompany', '证券公司债', '否'),
    ('securitiescompanycps', '证券公司短融债', '否'),
    ('otherfinancialagencybonds', '其他金融机构债', '否'),
    ('enterprisebonds', '企业债', '否'),
    ('enterprisebonds', '企业债', '是'),  # 城投
    ('corporatebonds', '公司债', '否'),
    ('corporatebonds', '公司债', '是'),   # 城投
    ('medium-termnotes', '中期票据', '否'),
    ('medium-termnotes', '中期票据', '是'),  # 城投
    ('cps', '短期融资券', '否'),
    ('cps', '短期融资券', '是'),  # 城投
    ('projectrevenuenotes', '项目收益票据', '否'),
    ('ppn', 'PPN', '否'),
    ('ppn', 'PPN', '是'),  # 城投
    ('supranationalbonds', '国际机构债', '否'),
    ('agencybonds', '政府支持机构债', '否'),
    ('standardizednotes', '标准化票据', '否'),
    ('abs', 'ABS', '否'),
    ('convertiblebonds', '可转债', '否'),
    ('exchangeablebonds', '可交换债', '否'),
    ('detachableconvertiblebonds', '可分离转债', '否'),
]

print(f"债券类型配置完成，共{len(BOND_TYPES)}类")

### 日志配置

In [ ]:
# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

def log_progress(message):
    """记录处理进度"""
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{current_time}] {message}")
    logger.info(message)

log_progress("日志系统初始化完成")

### 数据库连接

In [ ]:
def create_mysql_engine():
    """创建MySQL数据库连接引擎"""
    connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
    engine = create_engine(
        connection_string,
        poolclass=pool.NullPool,
        pool_recycle=3600,
        echo=False
    )
    return engine

# 创建连接
sql_engine = create_mysql_engine()
log_progress("MySQL数据库连接创建成功")

### 测试数据库连接

In [ ]:
# 测试MySQL连接
try:
    with sql_engine.connect() as conn:
        result = conn.execute(text("SELECT 1 as test"))
        print(f"MySQL连接测试成功: {result.fetchone()}")
except Exception as e:
    print(f"MySQL连接测试失败: {e}")

In [ ]:
# 检查必要的表是否存在
required_tables = ['债券到期', '债券新发行1']

try:
    inspector = inspect(sql_engine)
    for table in required_tables:
        exists = inspector.has_table(table)
        status = "存在" if exists else "不存在"
        print(f"表 {table}: {status}")
except Exception as e:
    print(f"表检查失败: {e}")

### 辅助函数

In [ ]:
def generate_column_mapping(columns):
    """
    生成列名映射（中文列名 -> col_0, col_1, ...）
    
    Args:
        columns: 列名列表
        
    Returns:
        列名映射字典
    """
    return {col: f"col_{i}" for i, col in enumerate(columns)}


def change_column_names(table_name, column_mapping, engine, to_english=True):
    """
    修改表列名（中文 <-> 英文编号）
    
    Args:
        table_name: 表名
        column_mapping: 列名映射字典
        engine: 数据库引擎
        to_english: True表示中文->英文，False表示英文->中文
    """
    with engine.begin() as connection:
        for original_name, new_name in column_mapping.items():
            if to_english:
                connection.execute(text(
                    f"ALTER TABLE `{table_name}` CHANGE `{original_name}` `{new_name}` VARCHAR(255);"
                ))
            else:
                connection.execute(text(
                    f"ALTER TABLE `{table_name}` CHANGE `{new_name}` `{original_name}` VARCHAR(255);"
                ))


def get_date_range(days_start=0, days_end=7, date_format='%Y-%m-%d'):
    """
    获取日期范围
    
    Args:
        days_start: 开始日期偏移（相对于今天）
        days_end: 结束日期偏移（相对于今天）
        date_format: 日期格式
        
    Returns:
        (开始日期字符串, 结束日期字符串)
    """
    current_date = datetime.now()
    start_date = current_date + timedelta(days=days_start)
    end_date = current_date + timedelta(days=days_end)
    return start_date.strftime(date_format), end_date.strftime(date_format)


print("辅助函数定义完成")

In [ ]:
# 测试辅助函数
print("列名映射测试:")
test_columns = ['dt', 'totalredemption', 'bondtype']
mapping = generate_column_mapping(test_columns)
print(f"  输入: {test_columns}")
print(f"  映射: {mapping}")

print("\n日期范围测试:")
dt0, dt1 = get_date_range(days_start=0, days_end=7)
print(f"  未来7天: {dt0} ~ {dt1}")

dt0, dt1 = get_date_range(days_start=1, days_end=30, date_format='%Y%m%d')
print(f"  未来1-30天: {dt0} ~ {dt1}")

### 配置验证

In [ ]:
def validate_config():
    """验证配置是否完整"""
    errors = []
    warnings = []
    
    if not MYSQL_PASSWORD:
        errors.append('MYSQL_PASSWORD 环境变量未设置')
    
    if not IFIND_USERNAME or not IFIND_PASSWORD:
        warnings.append('IFIND_USERNAME 或 IFIND_PASSWORD 环境变量未设置（债券新发行功能将不可用）')
    
    if not WIND_USERNAME or not WIND_PASSWORD:
        warnings.append('WIND_USERNAME 或 WIND_PASSWORD 环境变量未设置（债券到期功能可能不可用）')
    
    if errors:
        print('配置错误:')
        for error in errors:
            print(f'  - {error}')
        return False
    
    if warnings:
        print('配置警告:')
        for warning in warnings:
            print(f'  - {warning}')
    
    if not errors and not warnings:
        print('配置验证通过')
    
    return len(errors) == 0

validate_config()

In [ ]:
# 打印配置摘要
print('=== T34 债券新发行 配置摘要 ===')
print(f'MySQL主机: {MYSQL_HOST}:{MYSQL_PORT}')
print(f'MySQL数据库: {MYSQL_DATABASE}')
print(f'债券类型数量: {len(BOND_TYPES)}')
print(f'债券到期采集范围: 未来{BOND_MATURITY_CONFIG["days_ahead"]}天')
print(f'债券新发行采集范围: 未来{BOND_ISSUE_CONFIG["days_start"]}-{BOND_ISSUE_CONFIG["days_end"]}天')
print(f'Wind API配置: {"已配置" if WIND_USERNAME else "未配置"}')
print(f'iFinD API配置: {"已配置" if IFIND_USERNAME else "未配置"}')
print('=' * 40)

---

### 环境初始化完成

**已初始化**:
- 依赖库导入
- 配置参数加载
- 日志系统配置
- 数据库连接创建
- 辅助函数定义

---

**下一章**: [3_数据采集与ETL](3_数据采集与ETL.ipynb)